# 促銷分析｜優惠券與折扣效益評估

商業問題：  
優惠券使用與客單價影響分析  
分析方法：  
- 區分使用與未使用優惠券的訂單
- 比較兩組平均訂單金額與平均訂單量
- 分析優惠券使用與消費金額的關聯

In [12]:
SELECT case
       when Coupon_Code is null then '未使用優惠券'
       else '使用優惠券'
       end AS [優惠券狀態],
       ROUND(AVG(Order_Value),2) AS [平均訂單金額],
       ROUND(AVG(Quantity*1.0),2) AS [平均訂單量]
  FROM india_ecom.dbo.sales
 GROUP BY case
       when Coupon_Code is null then '未使用優惠券'
       else '使用優惠券'
       end;

(2 個資料列受到影響)

優惠券狀態  | 平均訂單金額   | 平均訂單量   
-------+----------+---------
未使用優惠券 | 23884.46 | 1.250000
使用優惠券  | 24299.94 | 1.250000
(2 個資料列)

總執行時間: 00:00:00.536

分析結果：  
使用優惠券的訂單平均金額為 24,299.94 ，略高於未使用優惠券的 23,884.46 ，差距約 415 元，幅度不大。兩組的平均訂單量皆為 1.25 ，優惠券使用與訂單金額存在差異，但在平均購買件數上未觀察到差異。

商業問題：  
各優惠券折扣效益與成本分析  
分析方法：  
- 計算各優惠券的折扣成本與帶動營收
- 比較不同優惠券的投入成本與營收效益
- 評估各優惠券的成本效益差異

In [13]:
SELECT Coupon_Code,
       ROUND(SUM(Coupon_Discount),2) AS [優惠券成本],
       ROUND(SUM(Order_Value),2) AS [優惠券訂單營收]
  FROM india_ecom.dbo.sales
 GROUP BY Coupon_Code
 ORDER BY [優惠券訂單營收] DESC;

(4 個資料列受到影響)

Coupon_Code | 優惠券成本       | 優惠券訂單營收      
------------+-------------+--------------
NULL        | 0           | 4772472864.35
SAVE10      | 60610436.52 | 606104289    
DIWALI100   | 1260900     | 308922640.44 
FLAT50      | 628950      | 304465761.41 
(4 個資料列)

總執行時間: 00:00:00.200

分析結果：  
無優惠券訂單帶動的訂單營收最高（約 47.7 億），顯示整體訂單營收主要來自未使用優惠券的訂單。三種優惠券中，SAVE10 帶動營收最高（約 6.06 億），但優惠券成本也最高（約 6,061 萬）； DIWALI100 與 FLAT50 的營收皆約 3 億出頭，但 DIWALI100 的優惠券成本約 126 萬，為 FLAT50 的兩倍，顯示兩者營收規模相近，但折扣投入存在明顯差異。


商業問題：  
促銷期間新舊客比例分析  
分析方法：  
- 找出每位客戶的首購日

In [16]:
SELECT TOP(5000)Customer_ID, ----因檔案過大，篩選前 5000 筆會員作為數據呈現，實際符合條件共 39726 人
       MIN(Order_Date) AS [首購日]
  FROM india_ecom.dbo.sales
 WHERE Order_Status='Delivered'
 GROUP BY Customer_ID
 ORDER BY [首購日];

(5000 個資料列受到影響)

Customer_ID  | 首購日       
-------------+-----------
CUST00009987 | 2024-06-01
CUST00029688 | 2024-06-01
CUST00015168 | 2024-06-01
CUST00028469 | 2024-06-01
CUST00026694 | 2024-06-01
CUST00025222 | 2024-06-01
CUST00003707 | 2024-06-01
CUST00033277 | 2024-06-01
CUST00005806 | 2024-06-01
CUST00018604 | 2024-06-01
CUST00036197 | 2024-06-01
CUST00016123 | 2024-06-01
CUST00034367 | 2024-06-01
CUST00033234 | 2024-06-01
CUST00037646 | 2024-06-01
CUST00012546 | 2024-06-01
CUST00032486 | 2024-06-01
CUST00013034 | 2024-06-01
CUST00035023 | 2024-06-01
CUST00009671 | 2024-06-01
CUST00018946 | 2024-06-01
CUST00037875 | 2024-06-01
CUST00035107 | 2024-06-01
CUST00009483 | 2024-06-01
CUST00037689 | 2024-06-01
CUST00038619 | 2024-06-01
CUST00009903 | 2024-06-01
CUST00034524 | 2024-06-01
CUST00037516 | 2024-06-01
CUST00027518 | 2024-06-01
CUST00016464 | 2024-06-01
CUST00035563 | 2024-06-01
CUST00032977 | 2024-06-01
CUST00001670 | 2024-06-01
CUST00023173 | 2024-06-01
CUST00034561 | 2024-0

分析結果：  
以 Customer_ID 分組，取每位客戶最早的 Order_Date 作為首購日。 25 萬筆訂單去重後共 39,914 位客戶，首購日分布於 2024/06/01 至 2026/06/28 。由於未加篩選條件，結果涵蓋所有訂單狀態。若要分析真實首購行為，建議加上 Order_Status 篩選出 Delivered 訂單。